# Alliane: hyperparameter experiments

Ten runs that change the learning rate and the batch size. Everything else
(gamma, epsilon schedule, buffer size, target sync, seed and the 400,000 step
budget) keeps the default from `train.py`, so the only difference between two
runs is the value we changed.

Each run saves `log.csv`, `model.pth` and `log.png` in its own folder,
`experiments/Alliane/expNN/`. Never rerun into a folder that already has results.

After each run, fill in its row in `AllianeHyperparameter_experiment_results.md`.

## Running on Google Colab

1. Open this notebook in Colab (File > Upload notebook).
2. Runtime > Change runtime type > T4 GPU.
3. Click the key icon in the left sidebar and add a secret named
   `GITHUB_TOKEN` with your GitHub token, and turn on Notebook access.
   Never paste the token into a cell.
4. Runtime > Run all. Keep the tab open and the laptop awake: Colab stops
   if you close the tab or leave it idle for too long.

Each run takes about 1.5 to 2 hours. Colab deletes its files when the session
ends, so each run is pushed to the `alliane` branch on GitHub as soon as it
finishes. Runs that are already on GitHub are skipped, so after a disconnect
you just press Run all again and it carries on with the next run.

## Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

BRANCH = "alliane"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata

    ROOT = Path("/content/dqn-atari")
    token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{token}@github.com/H-levison/dqn-atari.git"
    if not ROOT.exists():
        # check=False so an error can't print the URL with the token in it
        result = subprocess.run(["git", "clone", "-b", BRANCH, repo_url, str(ROOT)], capture_output=True)
        if result.returncode != 0:
            raise RuntimeError("git clone failed. Check the GITHUB_TOKEN secret.")
    subprocess.run(["git", "config", "user.name", "Alliane Umutoniwase"], cwd=ROOT, check=True)
    subprocess.run(["git", "config", "user.email", "a.umutoniwa1@alustudent.com"], cwd=ROOT, check=True)

    # Install uv (the group rule is uv, never pip)
    subprocess.run("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, check=True)
    os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
else:
    # Running locally: this notebook is in experiments/Alliane, train.py is two folders up
    ROOT = Path("../..").resolve()

subprocess.run(["uv", "sync"], cwd=ROOT, check=True)
subprocess.run(["uv", "run", "python", "-c", "import torch; print('GPU available:', torch.cuda.is_available())"], cwd=ROOT, check=True)

sys.path.insert(0, str(ROOT))
from plot_results import plot_training


def save_to_github(name):
    # Push one finished run so it survives the Colab session ending
    folder = f"experiments/Alliane/{name}"
    subprocess.run(["git", "add", folder], cwd=ROOT, check=True)
    subprocess.run(["git", "commit", "-m", f"Add Alliane {name} results"], cwd=ROOT, check=True)
    subprocess.run(["git", "pull", "--rebase", "origin", BRANCH], cwd=ROOT, capture_output=True)
    result = subprocess.run(["git", "push", "origin", BRANCH], cwd=ROOT, capture_output=True)
    if result.returncode == 0:
        print(name, "pushed to GitHub")
    else:
        print("Push failed for", name, "- download its folder from the Files panel instead")


def run_experiment(name, lr, batch_size):
    out = f"experiments/Alliane/{name}"
    already_done = (ROOT / out / "log.csv").exists()

    if already_done:
        print(out, "already has results, so it is not trained again.")
    else:
        command = ["uv", "run", "train.py", "--lr", str(lr), "--batch-size", str(batch_size), "--out", out]
        print("Running:", " ".join(command))
        subprocess.run(command, cwd=ROOT, check=True)

    # Plot the run and save the figure next to its log
    log = pd.read_csv(ROOT / out / "log.csv")
    fig = plot_training(log, title=f"{name}: lr={lr}, batch size={batch_size}")
    fig.savefig(ROOT / out / "log.png", dpi=120)
    plt.show()

    if IN_COLAB and not already_done:
        save_to_github(name)

    mean_100 = log["reward"].rolling(100, min_periods=1).mean()
    print(f"Episodes: {len(log)}")
    print(f"Mean reward of the last 100 episodes: {mean_100.iloc[-1]:.1f}")
    print(f"Best mean-100 reward during training: {mean_100.max():.1f}")

## Learning rate

Batch size stays at 32. The learning rate controls how big each update to the
weights is. Too small and the agent learns very slowly; too big and the
Q-values jump around or blow up.

In [ ]:
# Experiment 1: baseline, the group's default values
run_experiment("exp01", lr=1e-4, batch_size=32)

In [ ]:
# Experiment 2: 10 times smaller than the baseline.
# Expect very slow learning: the reward may stay near -21 for the whole run.
run_experiment("exp02", lr=1e-5, batch_size=32)

In [ ]:
# Experiment 3: half the baseline
run_experiment("exp03", lr=5e-5, batch_size=32)

In [ ]:
# Experiment 4: 2.5e-4 is the learning rate used in the original DQN paper
run_experiment("exp04", lr=2.5e-4, batch_size=32)

In [ ]:
# Experiment 5: 5 times the baseline. Faster learning, or the start of instability?
run_experiment("exp05", lr=5e-4, batch_size=32)

In [ ]:
# Experiment 6: 10 times the baseline. Expect unstable training:
# the loss jumps around and the reward may stop improving or get worse.
run_experiment("exp06", lr=1e-3, batch_size=32)

## Batch size

Learning rate stays at 1e-4. The batch size is how many transitions from the
replay buffer go into one update. Small batches give noisy updates; big batches
give smoother ones but each update takes longer.

In [ ]:
# Experiment 7: half the baseline batch
run_experiment("exp07", lr=1e-4, batch_size=16)

In [ ]:
# Experiment 8: double the baseline batch
run_experiment("exp08", lr=1e-4, batch_size=64)

In [ ]:
# Experiment 9: four times the baseline batch
run_experiment("exp09", lr=1e-4, batch_size=128)

## Combination

In [ ]:
# Experiment 10: a bigger batch gives a less noisy gradient, so it may handle
# a higher learning rate better than batch size 32 did in experiment 4
run_experiment("exp10", lr=2.5e-4, batch_size=64)

## Summary of all runs

Puts every finished run in one table, to help fill in the results file.

In [ ]:
rows = []
for folder in sorted((ROOT / "experiments/Alliane").glob("exp*")):
    csv_path = folder / "log.csv"
    if not csv_path.exists():
        continue
    log = pd.read_csv(csv_path)
    mean_100 = log["reward"].rolling(100, min_periods=1).mean()
    rows.append({
        "experiment": folder.name,
        "episodes": len(log),
        "final mean-100 reward": round(mean_100.iloc[-1], 1),
        "best mean-100 reward": round(mean_100.max(), 1),
        "final loss": round(log["loss"].iloc[-1], 4),
    })

pd.DataFrame(rows)

## What I found

Best and worst configuration, and which hyperparameter caused it.
This is the part I present.